# 🧠 Module 03: Generalization, Regularization & Normalization (2014 - 2019)
### *From Dropout and the BatchNorm Myth to Modern SOTA RMSNorm and AdamW*

---

## 📜 Historical Context & The Three Great Crises
Deep neural networks possess millions (and now trillions) of free parameters. With this immense capacity comes three existential challenges:
1. **The Overfitting Crisis (Co-adaptation)**: Neurons learn complex mutual dependencies on training data that fail miserably on test data.
2. **The Internal Covariate Shift Myth vs Loss Surface Smoothing**: Why does Batch Normalization actually work? (Santurkar et al. 2018 debunking Ioffe & Szegedy 2015).
3. **The Adaptive Regularization Breakdown**: Why standard L2 regularization fails with Adam, necessitating Loshchilov & Hutter's 2019 **AdamW** (Decoupled Weight Decay).

In this notebook, we journey from foundational principles to the modern normalization stack powering **LLaMA 3**, **Mistral**, and modern Transformers.

## 1. The Overfitting Crisis & Dropout (Srivastava et al., 2014)

### 🎻 The Musician Ensemble Analogy:
> If an orchestra always practices together, the second violinist may stop playing the difficult passages, relying on the first violinist to cover them (**co-adaptation**).
> But if the conductor randomly mutes 50% of the musicians on each measure during rehearsal, every single player is forced to master the score independently. At concert time (inference), all musicians play together, creating an immensely robust ensemble.

### Inverted Dropout Formulation:
During training, sample a Bernoulli mask $m \sim \text{Bernoulli}(1 - p)$ and scale activations by $\frac{1}{1-p}$:
$$y = \frac{x \odot m}{1 - p}$$
At inference time, dropout is disabled: $y = x$. No scaling or computational overhead is required at test time!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class InvertedDropout:
    def __init__(self, p: float = 0.5):
        self.p = p
        self.keep_prob = 1.0 - p
        self.training = True
        self.mask = None
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        if not self.training or self.p == 0.0:
            return x
        self.mask = (np.random.rand(*x.shape) < self.keep_prob).astype(np.float64)
        return (x * self.mask) / self.keep_prob

# Verify Expectation Preservation E[y] == E[x]
np.random.seed(42)
x = np.ones((1000, 50))
dropout = InvertedDropout(p=0.4)

y_train = dropout.forward(x)
dropout.training = False
y_test = dropout.forward(x)

print(f"Input mean:            {np.mean(x):.4f}")
print(f"Training output mean:  {np.mean(y_train):.4f} (Expectation preserved!)")
print(f"Dropout rate active:   {np.mean(y_train == 0.0):.4f} (target p=0.40)")
print(f"Inference output mean: {np.mean(y_test):.4f} (Zero-overhead identity!)")

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.title('Training Activations (p=0.4 Sparsified)')
plt.imshow(y_train[:20, :20], cmap='viridis')
plt.colorbar()
plt.subplot(1, 2, 2)
plt.title('Inference Activations (Dense Identity)')
plt.imshow(y_test[:20, :20], cmap='viridis')
plt.colorbar()
plt.tight_layout()
plt.show()

## 2. Decoupled Weight Decay vs L2 Regularization (Loshchilov & Hutter, 2019)

Why did standard $L_2$ regularization fail when deep learning moved from SGD to adaptive optimizers like Adam?

- **In SGD**, adding $\frac{1}{2}\lambda \|w\|^2$ to the loss yields:
  $$w_{t+1} = w_t - \eta (\nabla L(w_t) + \lambda w_t) = w_t(1 - \eta \lambda) - \eta \nabla L(w_t)$$
  *Here, $L_2$ regularization and weight decay are identical.*

- **In Adam**, $L_2$ modifies the gradient estimate: $g_t = \nabla L(w_t) + \lambda w_t$. This gradient is fed into the variance estimate $v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$:
  $$w_{t+1} = w_t - \frac{\eta}{\sqrt{v_t} + \epsilon} m_t$$
  *Catastrophe*: Parameters that receive frequent, large gradients have large $v_t$, which **divides and shrinks** the weight decay penalty! The parameters that need the most regularization end up receiving the least.

- **AdamW (Decoupled Weight Decay)** removes $\lambda w_t$ from the moments entirely, applying it directly:
  $$w_{t+1} = w_t - \eta \lambda w_t - \frac{\eta}{\sqrt{v_t} + \epsilon} m_t$$

In [ ]:
# Simulation: Tracking parameter decay under disparate gradient scales
w_adam_l2 = np.array([2.0, 2.0])
w_adam_w  = np.array([2.0, 2.0])

# Dimension 0 has huge gradients; Dimension 1 has sparse, small gradients
grads = np.array([50.0, 0.05])
lr = 0.01
decay = 0.1

# Adam moments
m_l2, v_l2 = np.zeros(2), np.zeros(2)
m_w, v_w = np.zeros(2), np.zeros(2)

trajectory_l2 = []
trajectory_w = []

for t in range(1, 100):
    # 1. Adam with L2
    g_l2 = grads + decay * w_adam_l2
    m_l2 = 0.9 * m_l2 + 0.1 * g_l2
    v_l2 = 0.999 * v_l2 + 0.001 * (g_l2**2)
    m_hat_l2 = m_l2 / (1 - 0.9**t)
    v_hat_l2 = v_l2 / (1 - 0.999**t)
    w_adam_l2 -= (lr / (np.sqrt(v_hat_l2) + 1e-8)) * m_hat_l2
    
    # 2. AdamW (Decoupled)
    g_w = grads
    m_w = 0.9 * m_w + 0.1 * g_w
    v_w = 0.999 * v_w + 0.001 * (g_w**2)
    m_hat_w = m_w / (1 - 0.9**t)
    v_hat_w = v_w / (1 - 0.999**t)
    w_adam_w -= lr * decay * w_adam_w + (lr / (np.sqrt(v_hat_w) + 1e-8)) * m_hat_w
    
    trajectory_l2.append(w_adam_l2.copy())
    trajectory_w.append(w_adam_w.copy())

trajectory_l2 = np.array(trajectory_l2)
trajectory_w = np.array(trajectory_w)

print(f"Final Adam + L2 weight (dim 0 with high grad): {trajectory_l2[-1, 0]:.4f}")
print(f"Final AdamW weight     (dim 0 with high grad): {trajectory_w[-1, 0]:.4f}")
print("Notice how AdamW strictly regularizes weights without moment distortion.")

## 3. The Normalization Revolution (BatchNorm vs LayerNorm vs RMSNorm)

### 🏭 The Assembly Line Quality Control Analogy:
If station 10 shifts components by +5mm, downstream station 50 drifts completely off calibration. Normalization locks the distribution into a clean coordinate space.

| Normalization | Normalizes Across | Batch Size Dependency | Primary Domain |
| :--- | :--- | :--- | :--- |
| **BatchNorm** (2015) | Mini-batch $(B)$ for each feature channel | High (fails if $B=1$) | Computer Vision (CNNs) |
| **LayerNorm** (2016) | Feature channels $(D)$ for each sample | Zero (works for $B=1$) | Recurrent Nets & Transformers |
| **RMSNorm** (2019) | Scale only (no mean centering) | Zero (works for $B=1$) | Modern LLMs (LLaMA 3, Mistral) |

### The Santurkar et al. (2018) Revelation:
Ioffe & Szegedy initially claimed BatchNorm accelerates training by reducing *Internal Covariate Shift (ICS)*. Santurkar et al. showed that even if you intentionally inject synthetic covariate shift into BatchNorm layers, the network trains just as fast! BatchNorm's real superpower is **smoothing the loss landscape**, making the gradient Lipschitz continuous and suppressing sharp ravine walls.

In [ ]:
import torch
import torch.nn as nn

# SOTA RMSNorm PyTorch Implementation (LLaMA 3 Style)
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast to float32 internally to prevent BF16/FP16 overflow
        variance = x.float().pow(2).mean(-1, keepdim=True)
        x_norm = x * torch.rsqrt(variance + self.eps)
        return (x_norm * self.weight).type_as(x)

# Distribution Comparison
torch.manual_seed(42)
raw_activations = torch.randn(64, 512) * 5.0 + 12.0  # severely shifted and scaled

bn = nn.BatchNorm1d(512)
ln = nn.LayerNorm(512)
rms = RMSNorm(512)

with torch.no_grad():
    act_bn = bn(raw_activations).flatten().numpy()
    act_ln = ln(raw_activations).flatten().numpy()
    act_rms = rms(raw_activations).flatten().numpy()
    act_raw = raw_activations.flatten().numpy()

plt.figure(figsize=(12, 4))
plt.hist(act_raw, bins=50, alpha=0.4, color='red', label=f'Raw (mean={act_raw.mean():.1f}, std={act_raw.std():.1f})')
plt.hist(act_bn, bins=50, alpha=0.5, color='blue', label=f'BatchNorm (mean={act_bn.mean():.2f}, std={act_bn.std():.2f})')
plt.hist(act_ln, bins=50, alpha=0.5, color='green', label=f'LayerNorm (mean={act_ln.mean():.2f}, std={act_ln.std():.2f})')
plt.hist(act_rms, bins=50, alpha=0.5, color='purple', label=f'RMSNorm (std={act_rms.std():.2f})')
plt.title('Live Distribution Transformation: Raw vs BatchNorm vs LayerNorm vs RMSNorm')
plt.xlabel('Activation Value')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Pre-LN vs Post-LN Architecture in Transformers

In the original 2017 Transformer (Vaswani et al.), **Post-LN** was used:
$$x = \text{LayerNorm}(x + \text{Sublayer}(x))$$
The normalization sits directly on the residual stream. As depth increases, gradients flowing backward through the normalization get progressively scaled down or degraded, requiring a delicate learning rate warmup schedule.

In **Pre-LN** (used in GPT-2, GPT-3, LLaMA, Mistral):
$$x = x + \text{Sublayer}(\text{RMSNorm}(x))$$
The residual stream $x$ is an unhindered **identity highway**, ensuring that gradients from the final loss can flow back directly to layer 1 without distortion!

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../python/module03_generalization_and_normalization'))
from pytorch_sota_norm import compare_pre_vs_post_ln_gradients

depths = [4, 8, 16, 24]
pre_grads, post_grads = [], []

for d in depths:
    g_pre, g_post = compare_pre_vs_post_ln_gradients(depth=d)
    pre_grads.append(g_pre)
    post_grads.append(g_post)

plt.figure(figsize=(8, 4))
plt.plot(depths, pre_grads, marker='o', linewidth=2, color='#06b6d4', label='Pre-LN (RMSNorm Highway)')
plt.plot(depths, post_grads, marker='s', linewidth=2, color='#f43f5e', label='Post-LN (Vaswani 2017)')
plt.title('Gradient Norm at Layer 1 vs Network Depth')
plt.xlabel('Number of Transformer Layers')
plt.ylabel('Input Gradient Norm ||dL/dx||')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Key Finding: Pre-LN preserves stable, well-behaved gradients across deep stacks,')
print('eliminating training collapse and enabling 100+ layer Transformer scaling!')